In [1]:
%pip install pyspark

Note: you may need to restart the kernel to use updated packages.


In [4]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("KafkaSparkStreaming").master("spark://localhost:7077").config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0").getOrCreate()
print(spark)

In [5]:
kafka_df = spark.readStream.format("kafka").option("kafka.bootstrap.servers", "172.17.0.1:9092").option("subscribe", "test").load()

print(kafka_df.printSchema())


root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)

None


In [ ]:
processed_df = kafka_df.select(F.col("key").cast("string"), F.col("value").cast("string"), F.col("topic"), F.col("timestamp"))
query = processed_df.writeStream.outputMode("append").format("console").option("truncate", "false").start()

query.awaitTermination()

In [ ]:
from pyspark.sql.types import StringType, StructField, StructType

json_schema = StructType(
    [
        StructField("firstName", StringType()),
        StructField("lastName", StringType())
    ]
)

json_df = kafka_df.select(F.col("key").cast("string"), F.col("value").cast("string"))
json_df = json_df.withColumn("jsonData", F.from_json(F.col("value"), json_schema)).select(F.col("jsonData.firstName"), F.col("jsonData.lastName"))

query = json_df.writeStream.format("console").outputMode("append").start()
query.awaitTermination()